# 문제 5 — 4x4 동차변환 모듈과 최소자승법

회전과 병진을 **한 행렬로 묶는** 것이 동차변환입니다.

$$T=\begin{bmatrix}R & \mathbf{t}\\ \mathbf{0}^{\mathsf{T}} & 1\end{bmatrix}\in\mathbb{R}^{4\times 4}$$

이렇게 묶으면 여러 좌표계를 지나는 변환을 **행렬 곱 하나로 연결**할 수 있습니다(문제 6).

## 이 노트북에서 해야 할 일

| # | 할 일 | 구현할 함수 |
|---|---|---|
| 5-1 | `make_T`, `inv_T` 를 만들고 ① 곱하면 단위행렬 ② 일반 역행렬과 일치 두 가지로 검증 | `make_T`, `inv_T` |
| 5-2 | **점(w=1)과 방향(w=0)의 차이**를 확인하고 왜 그런지 설명 | `to_homogeneous`, `transform_point`, `transform_direction`, `transform_points` |
| 5-3 | 두 변환의 **합성 순서**가 바뀌면 결과가 달라짐을 확인하고 3D 로 나란히 비교 | — |
| 5-4 | `inv_T` 와 일반 역행렬의 **실행 시간 비교**, 차이를 연산량 관점에서 설명 | `inv_T_batch` |
| 5-5 | **최소자승법** — 과결정 캘리브레이션 문제를 정규방정식으로 풀고 `lstsq` 와 비교, 잔차 정량 평가 | `least_squares_normal_equation`, `rmse` |

> `inv_T` 는 **일반 역행렬 함수를 쓰지 말고** 회전 부분의 전치를 이용한 공식으로 구현합니다.
> `tests/test_transform.py` 도 함께 작성해 제출합니다.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.rotation import rot_x, rot_y, rot_z
from src.transform import (inv_T, inv_T_batch, least_squares_normal_equation, make_T, rmse,
                           to_homogeneous, transform_direction, transform_point,
                           transform_points)
from src.vectors import det

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)

for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if _f in {f.name for f in __import__("matplotlib").font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False


def check(label, condition):
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


# --- 그림 헬퍼 (그대로 쓰면 됩니다) -----------------------------------------

def draw_frame(ax, T, scale=0.35, alpha=1.0, name=""):
    """동차변환 T 가 나타내는 좌표계를 그린다."""
    o = T[:3, 3]
    colors = ["r", "g", "b"]
    for i in range(3):
        v = T[:3, i] * scale
        ax.quiver(*o, *v, color=colors[i], alpha=alpha, arrow_length_ratio=0.18)
    if name:
        ax.text(*(o + 0.05), name, fontsize=9)


def setup_axes(ax, title, lim=1.0):
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title(title, fontsize=10)
    ax.set_box_aspect([1, 1, 1])


print("준비 완료")

## 5-1. `make_T` 와 `inv_T` — 역변환은 전치로 끝난다

역변환을 일반 역행렬로 구할 필요가 없습니다.
$R$ 이 직교행렬이라 $R^{-1}=R^{\mathsf{T}}$ 이기 때문입니다.

$$T^{-1}=\begin{bmatrix}R^{\mathsf{T}} & -R^{\mathsf{T}}\mathbf{t}\\ \mathbf{0}^{\mathsf{T}} & 1\end{bmatrix}$$

**유도해 보세요.** $T^{-1}$ 을 $\begin{bmatrix}S&\mathbf{u}\\0&1\end{bmatrix}$ 로 두고
$TT^{-1}=I$ 를 풀면 $S$ 와 $\mathbf{u}$ 가 각각 무엇이 되는지 나옵니다.
이 식이 기하적으로 무슨 뜻인지도 한 문장으로 적어 보세요.

### 역변환 공식의 유도와 의미

- 유도: `___`
- 의미: `___`

In [ ]:
R = rot_z(np.deg2rad(45.0)) @ rot_y(np.deg2rad(-30.0)) @ rot_x(np.deg2rad(15.0))
t = np.array([0.3, -0.2, 0.45])

# TODO: T = make_T(R, t), Ti = inv_T(T) 를 만들고
#       T, Ti, T @ Ti, np.linalg.inv(T) (# 검산용) 를 출력해 비교하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - ① T @ inv_T(T) == I, inv_T(T) @ T == I
#   - ② 일반 역행렬 np.linalg.inv 와 일치
#   - 역변환의 회전 부분도 회전행렬인가 (det = 1)
#   - 마지막 행이 (0,0,0,1) 인가
#   - inv_T(inv_T(T)) == T
#   - 무작위 200개 변환에서도 모두 성립하는가

## 5-2. 점과 방향의 차이 — 마지막 성분 1 vs 0

동차좌표의 네 번째 성분 $w$ 가 의미를 가릅니다.

$$T\begin{bmatrix}\mathbf{p}\\1\end{bmatrix}=?
\qquad\qquad
T\begin{bmatrix}\mathbf{v}\\0\end{bmatrix}=?$$

**할 일**

- 같은 벡터를 점(w=1)과 방향(w=0)으로 각각 변환해 결과를 비교하세요.
- 두 결과의 **차이가 무엇과 같은지** 확인하고, 왜 그렇게 되는지 식에서 설명하세요.
- 물리적으로 무엇이 점이고 무엇이 방향인지(위치 / 속도·힘·법선벡터),
  이걸 헷갈리면 로봇에서 어떤 문제가 생기는지 적으세요.
- 보너스: **두 점의 차이**는 점처럼 변환될까요, 방향처럼 변환될까요? 코드로 확인하세요.

### 점과 방향이 다르게 변환되는 이유

- 식에서의 이유: `___`
- 물리적 의미: `___`
- 헷갈리면 생기는 문제: `___`

In [ ]:
v = np.array([1.0, 0.0, 0.0])

# TODO: transform_point / transform_direction 결과를 출력하고 차이를 확인하세요.
# TODO: 길이가 보존되는 쪽은 어느 쪽인지도 출력해 보세요.
# TODO: 두 점의 차이 (p1 - p2) 를 방향으로 변환한 것과
#       변환된 p1 - 변환된 p2 를 비교하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 점과 방향의 결과가 다른가
#   - 그 차이가 정확히 병진 벡터 t 인가
#   - 방향은 길이가 보존되는가 / 방향 변환이 R @ v 인가
#   - 점 변환이 R @ p + t 인가
#   - 두 점의 차이는 방향처럼 변환되는가
#   - 원점을 점으로 변환하면 t 인가 / 영벡터를 방향으로 변환하면 영벡터인가

## 5-3. 합성 순서가 바뀌면 결과가 다르다

회전과 마찬가지로 동차변환의 곱도 교환법칙이 성립하지 않습니다.
곱을 블록 단위로 전개해 보면 **회전이 병진을 끌고 도는** 구조가 보입니다.

$$T_1T_2=\begin{bmatrix}?&?\\0&1\end{bmatrix}
\qquad
T_2T_1=\begin{bmatrix}?&?\\0&1\end{bmatrix}$$

**할 일**

- 두 변환 $T_1, T_2$ 를 잡아 두 순서로 합성하고 회전 부분·병진 부분을 각각 비교하세요.
  (주의: 병진 벡터가 회전축과 나란하면 순서를 바꿔도 병진이 우연히 같아집니다.
   두 축 모두에 성분이 걸리도록 잡으세요.)
- 전개한 공식이 실제 계산과 맞는지 코드로 확인하세요.
- 두 결과 좌표계를 **3D 로 나란히 그려** 비교하세요.

### 블록 전개와 해석

- $T_1T_2$ 의 회전/병진: `___`
- $T_2T_1$ 의 회전/병진: `___`
- 일상적인 비유로 설명하면: `___`

In [ ]:
T1 = make_T(rot_z(np.deg2rad(90.0)), [0.5, 0.2, 0.0])    # z 90도 회전 + (0.5, 0.2, 0) 이동
T2 = make_T(rot_x(np.deg2rad(90.0)), [0.1, 0.0, 0.4])    # x 90도 회전 + (0.1, 0, 0.4) 이동

# TODO: C12 = T1 @ T2, C21 = T2 @ T1 을 만들고
#       두 행렬, 회전 부분 차이, 병진 부분을 각각 출력하세요.
# TODO: 전개 공식(회전이 병진에 어떻게 걸리는지)이 맞는지 확인해 출력하세요.

In [ ]:
# TODO: 3분할 3D 그림을 그리세요.
#   ① 기준 좌표계와 T1, T2   ② T1 @ T2   ③ T2 @ T1
#   draw_frame / setup_axes 를 쓰고, 원점에서 각 좌표계 원점까지 점선을 그으면 비교가 쉽습니다.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 두 합성 결과가 다른가 / 병진 부분도 다른가
#   - 전개한 합성 공식이 맞는가
#   - 두 결과 모두 유효한 동차변환인가 (회전부 det = 1)
#   - 역변환 순서가 뒤집히는가: (T1 T2)^-1 == T2^-1 T1^-1

## 5-4. `inv_T` vs 일반 역행렬 — 연산량 비교

두 방법이 각각 무슨 일을 하는지 세어 보세요.

| 방법 | 하는 일 | 연산량 |
|---|---|---|
| `inv_T` | `___` | 약 `___` flops |
| `np.linalg.inv` | `___` | 약 `___` flops |

**주의** — 4x4 하나만 뒤집는 **단건 호출로는 이 차이가 잘 측정되지 않습니다.**
실제 연산은 수십 flops(나노초)뿐이고, 시간의 대부분은 파이썬 함수 호출과
NumPy 배열 생성 오버헤드(마이크로초)이기 때문입니다.

**할 일**

- 먼저 단건 호출로 측정해 보고, 결과가 예상과 다르면 **왜 그런지** 적으세요.
- `(N, 4, 4)` 묶음을 한 번에 뒤집는 `inv_T_batch` 를 구현해
  오버헤드를 상수화한 뒤 다시 비교하세요.
  (로봇 제어 루프에서 링크 수십 개의 변환을 매 주기 뒤집는 상황이 정확히 이 형태입니다)
- 정확도 관점의 차이도 하나 있습니다. 전치로 구한 역변환의 회전부 직교성 오차와
  일반 역행렬의 직교성 오차를 재서 비교해 보세요.

### 속도 차이의 해석

- 단건에서 차이가 나지 않는 이유: `___`
- 배치에서 차이가 드러나는 이유: `___`
- 정확도 차이: `___`

In [ ]:
N_REPEAT = 20000


def bench(fn, repeat):
    """워밍업 후 repeat 번 재서 최솟값을 쓴다. (그대로 쓰면 됩니다)"""
    fn()
    best = float("inf")
    for _ in range(repeat):
        t0 = time.perf_counter()
        fn()
        best = min(best, time.perf_counter() - t0)
    return best


# TODO: 단건 호출로 inv_T 와 np.linalg.inv 의 1회 시간을 재서 출력하세요.

In [ ]:
N_BATCH = 2000

# TODO: (N_BATCH, 4, 4) 변환 묶음을 만들고
#       inv_T_batch 와 np.linalg.inv(스택 입력) 의 시간을 비교 출력하세요.
# TODO: 두 방법의 회전부 직교성 오차도 비교 출력하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 배치에서 inv_T_batch 가 일반 역행렬보다 빠른가
#   - inv_T 와 일반 역행렬의 결과가 같은가
#   - inv_T_batch 결과 == np.linalg.inv 스택 결과
#   - inv_T_batch 결과 == inv_T 를 하나씩 적용한 결과
#   - inv_T 결과의 회전부가 완전한 직교인가

## 5-5. 최소자승법 — 과결정 캘리브레이션 문제

**상황**: 카메라와 로봇 base 양쪽에서 좌표를 아는 대응점 $N$ 개가 있고,
이 대응을 설명하는 변환 $M=[R\,|\,\mathbf{t}]$ (12개 미지수)를 찾고 싶습니다.
측정에는 노이즈가 섞여 있고 대응점은 $N=30$ 개라 **방정식이 $3N=90$ 개, 미지수가 12 개**인
과결정(overdetermined) 문제입니다. 정확히 만족하는 해는 없으므로 오차 제곱합을 최소화합니다.

각 대응점은 다음 3 개의 선형방정식을 줍니다.

$$\mathbf{p}_{base} = M\begin{bmatrix}\mathbf{p}_{cam}\\1\end{bmatrix}
\;\Longrightarrow\;
\underbrace{(I_3\otimes[\mathbf{p}_{cam}^{\mathsf{T}}\;1])}_{3\times 12}\,\mathrm{vec}(M)=\mathbf{p}_{base}$$

최소자승해는 **정규방정식** $A^{\mathsf{T}}A\,\mathbf{x}=A^{\mathsf{T}}\mathbf{b}$ 에서 나옵니다.
이 식이 어디서 나오는지(잔차와 $A$ 의 열공간 사이의 관계) 직접 유도해 적어 보세요.

**할 일**

- 참값 변환을 하나 정하고, 대응점 30개에 1 mm 정도의 노이즈를 섞어 데이터를 만드세요.
- 설계행렬 $A$ (3N x 12) 와 관측 $b$ (3N,) 를 조립하세요. (힌트: `np.kron`)
- `least_squares_normal_equation` 으로 풀고 `np.linalg.lstsq` (# 비교 대상) 와 비교하세요.
- 잔차를 정량 평가하세요 — RMSE 가 주입한 노이즈와 같은 자릿수인지,
  잔차가 정말 $A$ 의 열공간에 수직인지($|A^{\mathsf{T}}r|$).
- 잔차 분포 히스토그램과 대응점별 재투영 오차 막대그래프를 그리세요.

### 정규방정식의 유도

- `___`

In [ ]:
N_PTS = 30
NOISE = 1e-3          # 1 mm 측정 노이즈

# TODO: 참값 변환 T_true 와 대응점 P_cam, 노이즈를 섞은 P_base 를 만드세요.
# TODO: 설계행렬 A_ls 와 관측 b_ls 를 조립하고 shape / 과결정 여부 / rank 를 출력하세요.
# TODO: least_squares_normal_equation 으로 풀어 M_hat 을 복원하고 참값과 비교하세요.
# TODO: np.linalg.lstsq (# 비교 대상) 와 일치하는지, 잔차 RMSE 와 |A^T r| 을 출력하세요.

In [ ]:
# TODO: 그래프 2개를 그리세요.
#   왼쪽  : 잔차 분포 히스토그램 (단위 mm)
#   오른쪽: 대응점별 재투영 오차 막대그래프 + 주입 노이즈 기준선

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 과결정 문제인가 (식 개수 > 미지수 개수)
#   - 정규방정식 해가 np.linalg.lstsq 와 일치하는가
#   - 잔차가 A 의 열공간에 수직인가 (A^T r = 0)
#   - 추정한 M 이 참값에 충분히 가까운가
#   - 잔차 RMSE 가 주입 노이즈와 같은 자릿수인가
#   - (보너스) 해를 살짝 흔들면 잔차가 정말 커지는가 = 최소인가

## 답안 템플릿 정리

In [ ]:
summary = """
1. 역변환 검증: 단위행렬 여부 ___ / 일반 역행렬과 일치 ___
   - 사용한 공식: ___

2. 점 변환 결과: ___ / 방향 변환 결과: ___
   - 차이의 이유: ___
   - 두 결과의 차이가 무엇과 같은가: ___

3. 합성 순서 비교 그림: 위 3분할 그림 참조
   - T1@T2 병진 ___ vs T2@T1 병진 ___
   - 달라지는 이유: ___

4. inv_T 와 일반 역행렬 속도
   - 단건 호출 : ___ us / ___ us
   - 배치 ___ 개 : ___ ms / ___ ms
   - 차이의 이유: ___

5. 최소자승 해: ___
   - lstsq 와 일치: ___
   - 잔차 RMSE: ___ m (= ___ mm), 주입 노이즈 sigma = 1.0 mm
   - 잔차가 열공간에 수직임을 보인 값 |A^T r| = ___
"""
print(summary)